In [3]:
import os
import json
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM


class GroverEmbeddingExtractor:
    def __init__(self, model_name_or_path: str, device: torch.device, output_hidden_state: bool = True):
        """
        Инициализирует токенизатор и модель GROVER для извлечения эмбеддингов.
        Если output_hidden_state=True — возвращаем последние скрытые представления.
        """
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_name_or_path, output_hidden_states=output_hidden_state
        ).to(device)
        self.model.eval()

    def extract_embeddings(self, sequences: list[str], batch_size: int = 16) -> np.ndarray:
        """
        Для каждого батча последовательностей:
         - токенизируем (padding=True, truncation=True)
         - прогоняем через модель, получаем hidden_states
         - пулим по длине: mask-пуллинг (среднее по ненулевым токенам)
        Возвращаем np.ndarray размера (N, hidden_size).
        """
        all_embs = []
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i : i + batch_size]
            enc = self.tokenizer(
                batch_seqs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.tokenizer.model_max_length,
            ).to(self.device)

            with torch.no_grad():
                out = self.model(**enc)
                hidden = out.hidden_states[-1]  # (B, L, H)

            mask = enc["attention_mask"].unsqueeze(-1)  # (B, L, 1)
            summed = (hidden * mask).sum(dim=1)  # (B, H)
            counts = mask.sum(dim=1)  # (B, 1)
            pooled = summed / counts  # (B, H)

            all_embs.append(pooled.cpu().numpy())

        return np.vstack(all_embs)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)
train_ds, test_ds = ds["train"], ds["test"]

extractor = GroverEmbeddingExtractor("PoetschLab/GROVER", device=device)

PARAMS_LOGREG = {"max_iter": 1000, "random_state": 42}
PATH_TO_SAVE_OUTPUTS = "."
BATCH_SIZE = 16

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds["task"]), desc="Baseline"):
    tr = train_ds.filter(lambda x, t=task: x["task"] == t)
    te = test_ds.filter(lambda x, t=task: x["task"] == t)
    seqs_tr, y_tr = tr["sequence"], np.array(tr["label"])
    seqs_te, y_te = te["sequence"], np.array(te["label"])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        "accuracy": float(accuracy_score(y_te, preds)),
        "f1_score": float(f1_score(y_te, preds, average="macro")),
    }

    with open(f"{PATH_TO_SAVE_OUTPUTS}/results_grover_task-{task}_baseline.json", "w") as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train["task"]), desc="Few-shot"):
        tr = train.filter(lambda x, t=task: x["task"] == t)
        te = test.filter(lambda x, t=task: x["task"] == t)
        seqs_tr, y_tr = tr["sequence"], np.array(tr["label"])
        seqs_te, y_te = te["sequence"], np.array(te["label"])

        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())

                X_k = extractor.extract_embeddings(
                    [seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE
                )
                y_k = y_tr[idxs]

                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te

                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average="macro"))

            res[task][k] = {
                "accuracy": float(np.mean(accs)),
                "f1_score": float(np.mean(f1s)),
            }

            with open(
                f"{PATH_TO_SAVE_OUTPUTS}/results_grover_task-{task}_k-{k}.json",
                "w",
            ) as f:
                json.dump(res, f, indent=4)

    return res

results_kshot = few_shot(train_ds, test_ds)

output = {"full": baseline, "kshot": results_kshot, "params": PARAMS_LOGREG}

with open(f"{PATH_TO_SAVE_OUTPUTS}/results_grover.json", "w") as f:
    json.dump(output, f, indent=4)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'InstaDeepAI/nucleotide_transformer_downstream_tasks' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some weights of the model checkpoint at PoetschLab/GROVER were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Few-shot: 100%|██████████| 1